In [18]:
import sys
import os
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader  
import yaml
from sr_model import paired_sr, single_sr
from dataset import omic_data,single_data
import torch
import muon as mu 
import json 
import scanpy as sc 

'''
train a paired model without pretraining weights
'''
def load_data():
    mdata = mu.read_h5mu('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/notebook/eval_data/M_rna_1/mdata.h5mu')
    
    rna = mdata['rna_count']
    gadata = mdata['ga_count'] 

    mutual_gene = np.load('/home/rsun@ZHANGroup.local/atac_pretrain/src/mutual_gene.npy', allow_pickle=True)
    gadata = gadata[:,mutual_gene]
    
    # lo1p transform 

    sc.pp.normalize_total(gadata, target_sum= 1e4)
    sc.pp.log1p(gadata)

    sc.pp.normalize_total(rna, target_sum= 1e4)
    sc.pp.log1p(rna)

    # feature selection

    sc.pp.highly_variable_genes(rna, n_top_genes= 5000)
    rna = rna[:,rna.var['highly_variable']]

    sc.pp.highly_variable_genes(gadata, n_top_genes= 10000)
    gadata = gadata[:,gadata.var['highly_variable']] 

    print(rna.shape, gadata.shape)

    #train_idx, test_idx = train_test_split(np.arange(N), test_size = 0.1, random_state = 42)
    train_idx = np.load('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/sr_result/train_test_split/train_id_1.npy', allow_pickle = True)
    test_idx = np.load('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/sr_result/train_test_split/test_id_1.npy', allow_pickle = True)

    rna_train, rna_test = rna[train_idx,:], rna[test_idx,:]
    gadata_train, gadata_test = gadata[train_idx,:], gadata[test_idx,:]
    return rna_train, rna_test, gadata_train, gadata_test

def process_data(rna_train, rna_test, gadata_train, gadata_test):

    train_rna = rna_train.X.toarray().astype(np.float32)
    train_ga = gadata_train.X.toarray().astype(np.float32)
    train_rna = torch.from_numpy(train_rna)
    train_ga = torch.from_numpy(train_ga)

    test_rna = rna_test.X.toarray().astype(np.float32)
    test_ga = gadata_test.X.toarray().astype(np.float32)
    test_rna = torch.from_numpy(test_rna)
    test_ga = torch.from_numpy(test_ga)
    return train_rna, train_ga, test_rna, test_ga

def set_rna_config(N, B=1024):
    # N is the dataset size , B is the batch size 

    if N >= 100000:
        print('Large dataset, please use pair_train.py')
        return None 
    steps = int(40*N/B) # at least 1000 steps 


    config = {
        'omic': {'model_type': 'rna'},

        'network': {
            'feature_num': 5000,
            'hidden_dims': [512, 256, 128],
            'dropout': 0.1,
            'layernorm_eps': 1e-8,
            'activation': 'leaky_relu',
            'input_dropout': 0.2,
            'vae_weight': 0,
            'ce_weights': None,
            'class_dict': None
        },
        'optimizer': {
            'learning_rate': 1e-4,
            'weight_decay': 0.01,
            'warmup_steps': 100,
            'anneal_steps': steps,
            'min_lr': 1e-6
        },
        'training': {
            'device': 'cuda',
            'training_steps': steps,
            'eval_steps': 100000,
            'save_steps': 100000, # set large, do not save checkpoint during training
            'log_dir': 'logs',
            'save_dir': 'saved_models',
            'run_name': 'tiny_rna'
        }
    }
    return config

def set_ga_config(N, B=1024):
    # N is the dataset size , B is the batch size 

    if N >= 100000:
        print('Large dataset, please use pair_train.py')
        return None 
    steps = int(60*N/B) # at least 1000 steps 


    config = {
        'omic': {'model_type': 'ga'},

        'network': {
            'feature_num': 10000,
            'hidden_dims': [512, 256, 128],
            'dropout': 0.1,
            'layernorm_eps': 1e-8,
            'activation': 'leaky_relu',
            'input_dropout': 0.2,
            'vae_weight': 0,
            'ce_weights': None,
            'class_dict': None
        },
        'optimizer': {
            'learning_rate': 1e-4,
            'weight_decay': 0.01,
            'warmup_steps': 100,
            'anneal_steps': steps,
            'min_lr': 1e-6
        },
        'training': {
            'device': 'cuda',
            'training_steps': steps,
            'eval_steps': 100000,
            'save_steps': 100000, # set large, do not save checkpoint during training
            'log_dir': 'logs',
            'save_dir': 'saved_models',
            'run_name': 'tiny_ga'
        }
    }
    return config
def rna_train(rna_data):
    print(1)
    rna_dataset = single_data(rna_data)
    print(rna_dataset)
    rna_loader = DataLoader(rna_dataset, batch_size = 1024, shuffle= True)
    for batch in rna_loader:
        print(batch)
        break

    '''
    ini config
    '''
    config = set_rna_config(N = rna_data.shape[0], B = 1024)
    print(config)

    '''
    set rna model 
    '''
    rna_model = single_sr(config)
    #rna_model.set_optimizer()
    rna_model.train_model(train_loader = rna_loader,save_config= False)
    return rna_model, config

def ga_train(ga_data):
    ga_dataset = single_data(ga_data)
    ga_loader = DataLoader(ga_dataset, batch_size = 1024, shuffle= True)

    '''
    ini config
    '''
    config = set_ga_config(N = ga_data.shape[0], B = 1024)

    '''
    set rna model 
    '''
    ga_model = single_sr(config)
    ga_model.set_optimizer()
    ga_model.train_model(train_loader = ga_loader)
    return ga_model, config 


In [2]:
rna_train, rna_test, gadata_train, gadata_test = load_data()
train_rna, train_ga, test_rna, test_ga = process_data(rna_train, rna_test, gadata_train, gadata_test)
print(train_rna.shape, train_ga.shape)
print(test_rna.shape, test_ga.shape)

combined_rna = torch.cat((train_rna, test_rna), dim=0)
combined_ga = torch.cat((train_ga, test_ga), dim=0)
print(combined_rna.shape, combined_ga.shape)

#rna_dataset = single_data(combined_rna)
#ga_dataset = single_data(combined_ga)

"""
prepare  multiomic data
"""
traindata = omic_data(train_rna, train_ga)
testdata = omic_data(test_rna, test_ga) 
print('dataset over')

'''
prepare dataloader 
'''
batchsize = 2048

train_loader = DataLoader(traindata, batch_size= batchsize, shuffle=True)
test_loader = DataLoader(testdata, batch_size= batchsize, shuffle=True)
print('dataloader over')

/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. U

(71002, 5000) (71002, 10000)
torch.Size([63901, 5000]) torch.Size([63901, 10000])
torch.Size([7101, 5000]) torch.Size([7101, 10000])
torch.Size([71002, 5000]) torch.Size([71002, 10000])
dataset over
dataloader over


In [ ]:
all_res = {}
for step_key in [20,40,60,80,100,120,140,160,180,200,220,240]:
    checkpoint_path = checkpoint_path = f'/home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_{100*step_key}.pth'

    with open('/home/rsun@ZHANGroup.local/sr_project/configs/paired_configs/config_scratch.yaml', 'r') as f:
        config = yaml.safe_load(f)

    ga_config = set_ga_config(N = test_ga.shape[0])
    rna_config = set_rna_config(N = test_rna.shape[0])


    paired_model = paired_sr(config,
                            rna_config = rna_config,
                            ga_config = ga_config,
                            sr_rna = None,
                            sr_ga = None)


    paired_model.load_checkpoint(checkpoint_path) 
    paired_model.model.to('cuda')

    rna_embed_list = []
    ga_embed_list = []

    eval_dic = {}
    eval_count = 0
    paired_model.model.eval()

    device = 'cuda'

    with torch.no_grad():
        for batch in train_loader:
            rna = batch['rna'].to(device)
            ga = batch['ga'].to(device)
            N = rna.shape[0] 
            eval_count += N 

            outputs = paired_model.model(rna, ga)
            for key in outputs:
                if 'loss' in key:
                    if key not in eval_dic:
                        eval_dic[key] = 0
                    if type(outputs[key]) == int:
                        eval_dic[key] += outputs[key] * N
                    else:
                        eval_dic[key] += outputs[key].item() * N 
            rna_embed = outputs['rna_embed'].detach().cpu().numpy() 
            ga_embed = outputs['ga_embed'].detach().cpu().numpy()
            rna_embed_list.append(rna_embed)
            ga_embed_list.append(ga_embed)

    for key in eval_dic:
        eval_dic[key] = eval_dic[key] / eval_count

    rna_embed = np.concatenate(rna_embed_list)
    ga_embed = np.concatenate(ga_embed_list) 

    eval_res = {}
    for k in [2,5,10,15,20,30,50,100]:
        tmp_s = calculate_hit_rate(rna_embed, ga_embed, K = k, metric = 'cosine')
        eval_res[f'top_{k}'] = tmp_s
    
    acc, matchscore, foscttm = matching_metrics( x=rna_embed, y=ga_embed, metric='cosine')
    eval_res['acc'] = acc
    eval_res['matchscore'] = matchscore
    eval_res['foscttm'] = foscttm
    print(eval_res)

    all_res[step_key*100] = eval_res


Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_2000.pth at step 2000
{'top_2': 0.0355583720602732, 'top_5': 0.10660470356287846, 'top_10': 0.17849598648077736, 'top_15': 0.23334741585692156, 'top_20': 0.2777777777777778, 'top_30': 0.34157160963244615, 'top_50': 0.42501056189269115, 'top_100': 0.5399943669905647, 'acc': 0.06386424601078033, 'matchscore': 0.06830023974180222, 'foscttm': 0.029937212355434895}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_4000.pth at step 4000
{'top_2': 0.030629488804393746, 'top_5': 0.08970567525700605, 'top_10': 0.15159836642726376, 'top_15': 0.1966624419095902, 'top_20': 0.23412195465427404, 'top_30': 0.2884804957048303, 'top_50': 0.3643852978453739, 'top_100': 0.4714124771158992, 'acc': 0.05344317853450775, 'matchscore': 0.0560484454035759, 'foscttm': 0.035231415182352066}
Initialize rna model
Initialize ga model
Initialize multi model


...

Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_20000.pth at step 20000
{'top_2': 0.017039853541754683, 'top_5': 0.0572454583861428, 'top_10': 0.10287283481199831, 'top_15': 0.1348401633572736, 'top_20': 0.16258273482608082, 'top_30': 0.20595690747782003, 'top_50': 0.2696803267145472, 'top_100': 0.37213068581889874, 'acc': 0.03295310586690903, 'matchscore': 0.03534713387489319, 'foscttm': 0.04790201969444752}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_22000.pth at step 22000
{'top_2': 0.018377693282636248, 'top_5': 0.05682298267849599, 'top_10': 0.10097169412758766, 'top_15': 0.13413603717786227, 'top_20': 0.1613153077031404, 'top_30': 0.2056752570060555, 'top_50': 0.26996197718631176, 'top_100': 0.3708632586959583, 'acc': 0.03281228244304657, 'matchscore': 0.03407970815896988, 'foscttm': 0.04838484153151512}
Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_24000.pth at step 24000
{'top_2': 0.01689902830587241, 'top_5': 0.05520349246584988, 'top_10': 0.09970426700464723, 'top_15': 0.13392479932403886, 'top_20': 0.15807632727784818, 'top_30': 0.20194338825517533, 'top_50': 0.2665821715251373, 'top_100': 0.36628643852978454, 'acc': 0.03231939300894737, 'matchscore': 0.03407970815896988, 'foscttm': 0.049231696873903275}


In [22]:
pd.DataFrame(all_res)

,2000,4000,6000,8000,10000,12000,14000,16000,18000,20000,22000,24000
top_2,0.035558,0.030629,0.028658,0.023236,0.023729,0.020490,0.019927,0.019152,0.017814,0.017040,0.018378,0.016899
top_5,0.106605,0.089706,0.082383,0.074497,0.072103,0.064780,0.061611,0.058654,0.058442,0.057245,0.056823,0.055203
top_10,0.178496,0.151598,0.139487,0.129700,0.123222,0.113505,0.109914,0.108435,0.104070,0.102873,0.100972,0.099704
top_15,0.233347,0.196662,0.182017,0.167019,0.157020,0.150472,0.144557,0.139910,0.137234,0.134840,0.134136,0.133925
top_20,0.277778,0.234122,0.214899,0.199056,0.186946,0.180538,0.175891,0.167300,0.165188,0.162583,0.161315,0.158076
top_30,0.341572,0.288480,0.266582,0.250176,0.235460,0.229827,0.222856,0.214618,0.210182,0.205957,0.205675,0.201943
top_50,0.425011,0.364385,0.340023,0.321856,0.306154,0.300310,0.291227,0.279749,0.274398,0.269680,0.269962,0.266582
top_100,0.539994,0.471412,0.447261,0.426912,0.413322,0.403816,0.394663,0.383960,0.376848,0.372131,0.370863,0.366286
acc,0.063864,0.053443,0.049218,0.042177,0.041614,0.037178,0.035065,0.035136,0.033516,0.032953,0.032812,0.032319
matchscore,0.068300,0.056048,0.049993,0.044219,0.041684,0.039431,0.037037,0.036896,0.035206,0.035347,0.034080,0.034080


In [25]:
#checkpoint_path = f'/home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_2000.pth'

with open('/home/rsun@ZHANGroup.local/sr_project/configs/paired_configs/config_scratch.yaml', 'r') as f:
    config = yaml.safe_load(f)

ga_config = set_ga_config(N = test_ga.shape[0])
rna_config = set_rna_config(N = test_rna.shape[0])


paired_model = paired_sr(config,
                        rna_config = rna_config,
                        ga_config = ga_config,
                        sr_rna = None,
                        sr_ga = None)

checkpoint_path = '/home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_2000.pth'
paired_model.load_checkpoint(checkpoint_path) 
paired_model.model.to('cuda')

rna_embed_list = []
ga_embed_list = []

eval_dic = {}
eval_count = 0
paired_model.model.eval()

device = 'cuda'

with torch.no_grad():
    for batch in test_loader:
        rna = batch['rna'].to(device)
        ga = batch['ga'].to(device)
        N = rna.shape[0] 
        eval_count += N 

        outputs = paired_model.model(rna, ga)
        for key in outputs:
            if 'loss' in key:
                if key not in eval_dic:
                    eval_dic[key] = 0
                if type(outputs[key]) == int:
                    eval_dic[key] += outputs[key] * N
                else:
                    eval_dic[key] += outputs[key].item() * N 
        rna_embed = outputs['rna_embed'].detach().cpu().numpy() 
        ga_embed = outputs['ga_embed'].detach().cpu().numpy()
        rna_embed_list.append(rna_embed)
        ga_embed_list.append(ga_embed)

    for key in eval_dic:
        eval_dic[key] = eval_dic[key] / eval_count
    print(eval_dic)


Initialize rna model
Initialize ga model
Initialize multi model


/home/rsun@ZHANGroup.local/sr_project/src/sr_model.py:431: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /home/rsun@ZHANGroup.local/sr_project/saved_models/paired_config_scratch_2025-02-24-15-04/checkpoint_2000.pth at step 2000
{'loss': 2.4078448583602636, 'clip_loss': 4.414931363311389, 'ga_loss': 0.10130695360107432, 'rna_loss': 0.09907216408117239, 'rna_kld_loss': 0.0, 'ga_kld_loss': 0.0}


In [28]:
rna_embed = np.concatenate(rna_embed_list)
ga_embed = np.concatenate(ga_embed_list)    

In [29]:
for k in [2,5,10,15,20,30,50,100]:
    print(calculate_hit_rate(rna_embed, ga_embed, K = k, metric = 'cosine'))
    #print(calculate_hit_rate(rna_embed, ga_embed, K = k, metric = 'euclidean'))

0.0355583720602732
0.10660470356287846
0.17849598648077736
0.23334741585692156
0.2777777777777778
0.34157160963244615
0.42501056189269115
0.5399943669905647


In [4]:
import torch
import scipy.spatial
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.neighbors import NearestNeighbors

def matching_metrics(similarity=None, x=None, y=None, metric='euclidean', **kwargs):
    """
    计算匹配指标。

    参数:
        similarity (torch.Tensor, optional): 预计算的相似性矩阵。
        x (np.ndarray, optional): 第一个模态的嵌入向量。
        y (np.ndarray, optional): 第二个模态的嵌入向量。
        metric (str): 距离度量方式，支持 'euclidean' 或 'cosine'。
        **kwargs: 其他参数传递给距离计算函数。

    返回:
        acc: 准确率。
        matchscore: 匹配分数。
        foscttm: FOSCTTM 指标。
    """
    if similarity is None:
        if x.shape != y.shape:
            raise ValueError("Shapes do not match!")
        
        if metric == 'euclidean':
            # 计算欧式距离矩阵并转换为相似性矩阵
            distance_matrix = scipy.spatial.distance_matrix(x, y, **kwargs)
            similarity = 1 - distance_matrix
        elif metric == 'cosine':
            # 计算余弦相似性矩阵
            similarity = cosine_similarity(x, y)
        else:
            raise ValueError("Unsupported metric. Choose 'euclidean' or 'cosine'.")
    
    if not isinstance(similarity, torch.Tensor):
        similarity = torch.from_numpy(similarity)

    with torch.no_grad():
        batch_size = similarity.shape[0]
        
        # 计算 acc_x 和 acc_y
        acc_x = (
            torch.sum(
                torch.argmax(similarity, dim=1)
                == torch.arange(batch_size).to(similarity.device)
            )
            / batch_size
        )
        acc_y = (
            torch.sum(
                torch.argmax(similarity, dim=0)
                == torch.arange(batch_size).to(similarity.device)
            )
            / batch_size
        )
        
        # 计算 foscttm_x 和 foscttm_y
        foscttm_x = (
            (similarity > torch.diag(similarity)).float().mean(axis=1).mean().item()
        )
        foscttm_y = (
            (similarity > torch.diag(similarity)).float().mean(axis=0).mean().item()
        )
        
        # 计算 matchscore
        X = similarity
        mx = torch.max(X, dim=1, keepdim=True).values
        hard_X = (mx == X).float()
        logits_row_sums = hard_X.clip(min=0).sum(dim=1)
        matchscore = hard_X.clip(min=0).diagonal().div(logits_row_sums).mean().item()

        # 计算平均值
        acc = (acc_x + acc_y) / 2
        foscttm = (foscttm_x + foscttm_y) / 2
        
        return acc.item(), matchscore, foscttm

def calculate_hit_rate(rna_embeddings, atac_embeddings, K, metric = 'euclidean'):
    N = rna_embeddings.shape[0]
    # 合并嵌入和生成标签
    combined = np.concatenate([rna_embeddings, atac_embeddings], axis=0)
    cell_ids = np.concatenate([np.arange(N), np.arange(N)])  # 细胞ID
    modalities = np.array(['RNA']*N + ['ATAC']*N)  # 模态标签
    
    # 计算K近邻
    if metric == 'euclidean':
        #print(f'metric used {metric}')
        nbrs = NearestNeighbors(n_neighbors=K, metric='euclidean').fit(combined)
    elif metric == 'cosine':
        #print(f'metric used {metric}')
        nbrs = NearestNeighbors(n_neighbors=K, metric='cosine').fit(combined)
    else:
        print(f'metric used not support')
    _, indices = nbrs.kneighbors(combined)
    
    hit_count = 0
    for i in range(2*N):
        current_cell = cell_ids[i]
        current_modality = modalities[i]
        # 检查每个邻居
        for neighbor_idx in indices[i]:
            # 排除自身（若K包含自身需处理）
            if neighbor_idx == i:
                continue
            neighbor_cell = cell_ids[neighbor_idx]
            neighbor_modality = modalities[neighbor_idx]
            # 命中条件：同一细胞且不同模态
            if neighbor_cell == current_cell and neighbor_modality != current_modality:
                hit_count += 1
                break  # 至少一个命中即停止
    
    hit_rate = hit_count / (2 * N)
    return hit_rate
